In [ ]:
# Install AWS CLI if missing
!pip install awscli

# Install pandas and pyarrow if needed
!pip install pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: docutils
    Found existing installation: docutils 0.21.2
    Uninstalling docutils-0.21.2:
      Successfully uninstalled docutils-0.21.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.


In [ ]:
import pandas as pd

In [ ]:
pd.set_option("display.max_colwidth", None)


In [ ]:
!aws s3 ls --no-sign-request s3://eotarchive/eot-index/table/eot-main/


                           PRE crawl=EOT-2004/
                           PRE crawl=EOT-2008/
                           PRE crawl=EOT-2012/
                           PRE crawl=EOT-2016/
                           PRE crawl=EOT-2020/


In [ ]:
# Show Parquet part files for EOT-2012
!aws s3 ls --no-sign-request s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/

2023-11-08 16:58:32  427456938 part-00000-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:32  336915049 part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:33  289150555 part-00002-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:34  310154264 part-00003-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:34  305666378 part-00004-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:40  302693565 part-00005-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:41  284224418 part-00006-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:41  263959485 part-00007-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:41  246432635 part-00008-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:42  244658321 part-00009-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
2023-11-08 16:58:47  253612283 part-00010-9a26f426

In [ ]:
!aws s3 ls --no-sign-request s3://eotarchive/crawl-data/EOT-2012/cdx.paths.gz

2022-04-06 18:33:28     696686 cdx.paths.gz


In [ ]:
!mkdir EOT-2012/parquet/cdx/

In [ ]:
!aws s3 cp --no-sign-request s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/EOT-2012-20120915074514-00001.cdxj.gz EOT-2012/parquet/cdx


download: s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/EOT-2012-20120915074514-00001.cdxj.gz to EOT-2012/parquet/cdx/EOT-2012-20120915074514-00001.cdxj.gz


In [ ]:
import gzip
import json
import pandas as pd

path_gzip_file = "/content/EOT-2012/parquet/cdx/EOT-2012-20120915074514-00001.cdxj.gz"

rows = []

with gzip.open(path_gzip_file, "rt", encoding="utf-8") as f:
    for line in f:
        try:
            key, ts, json_blob = line.strip().split(" ", 2)
            record = json.loads(json_blob)

            record["surtkey"] = key
            record["timestamp"] = ts

            rows.append(record)

        except Exception as e:
            print("Parse error:", e)

df = pd.DataFrame(rows)


In [ ]:
df.iloc[0]

,0
url,http://www.afcrossroads.com/communications/images/include_pic.jpg
mime,image/jpeg
status,200
digest,sha1:Q3PRZVZH2Q2X7U3KJGXHAIF3J5QRX5YC
length,3873
offset,136238859
filename,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2012-20120915074514-00001.warc.gz
mime-detected,image/jpeg
puid,fmt/44
surtkey,"com,afcrossroads)/communications/images/include_pic.jpg"


In [ ]:
df.nunique()

,0
url,21047
mime,40
status,10
digest,17197
length,10673
offset,21067
filename,1
mime-detected,26
puid,49
surtkey,20419


In [ ]:
path_gzip_file = "/content/EOT-2012/parquet/cdx/cdx.paths.gz"

df = pd.read_csv(
    path_gzip_file, compression="gzip", header=0, sep=",", quotechar='"'
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/EOT-2012/parquet/cdx/cdx.paths.gz'

In [ ]:
import pandas as pd

# Load the downloaded Parquet file
df = pd.read_parquet("EOT-2012/parquet/subset/part-00047-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet")

# Show column names and types
# print(df.dtypes)

# Preview the first few rows
df.loc[0]

,0
url_surtkey,"gov,gpo)/fdsys/search/pagedetails.action?collapse=true&collectioncode=uscode&frompagedetails=true&granuleid=uscode-2010-title25-chap1-sec4&oldpath=title+25/chapter+1/sec.+4&packageid=uscode-2010-title25&searchpath=title+25/chapter+40"
url,http://www.gpo.gov/fdsys/search/pagedetails.action?collectionCode=USCODE&searchPath=Title+25%2FCHAPTER+40&granuleId=USCODE-2010-title25-chap1-sec4&packageId=USCODE-2010-title25&oldPath=Title+25%2FChapter+1%2FSec.+4&fromPageDetails=true&collapse=true
url_host_name,www.gpo.gov
url_host_tld,gov
url_host_2nd_last_part,gpo
url_host_3rd_last_part,www
url_host_4th_last_part,None
url_host_5th_last_part,None
url_host_registry_suffix,gov
url_host_registered_domain,gpo.gov


In [ ]:
!mkdir -p EOT-2012/parquet/subset/


In [ ]:
!aws s3 cp --no-sign-request s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet EOT-2012/parquet/subset/


download: s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet to EOT-2012/parquet/subset/part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet


In [ ]:
import pandas as pd

# Load the downloaded Parquet file
df = pd.read_parquet("EOT-2012/parquet/subset/part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet")

# Show column names and types
# print(df.dtypes)

# Preview the first few rows
df.nunique()

,0
url_surtkey,2808582
url,2839383
url_host_name,3297
url_host_tld,4
url_host_2nd_last_part,1266
url_host_3rd_last_part,1670
url_host_4th_last_part,130
url_host_5th_last_part,18
url_host_registry_suffix,4
url_host_registered_domain,1266


In [ ]:
import duckdb
import pandas as pd

In [ ]:
df = duckdb.sql("SELECT * FROM 'EOT-2012/parquet/subset/part-00047-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet' LIMIT 5").df()


In [ ]:
pd.DataFrame(df.columns, columns=["Column Name"])


,Column Name
0,url_surtkey
1,url
2,url_host_name
3,url_host_tld
4,url_host_2nd_last_part
5,url_host_3rd_last_part
6,url_host_4th_last_part
7,url_host_5th_last_part
8,url_host_registry_suffix
9,url_host_registered_domain


In [ ]:
duckdb.sql("""
    SELECT url_surtkey, fetch_status
    FROM 'EOT-2012/parquet/subset/part-00047-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet'
    WHERE fetch_status = 404
    LIMIT 5
""").df()

,url,fetch_status
0,http://www.gpo.gov/fdsys/search/pagedetails.ac...,404
1,http://www.gpo.gov/fdsys/search/pagedetails.ac...,404
2,http://www.gpo.gov/fdsys/search/pagedetails.ac...,404
3,http://www.gpo.gov/fdsys/search/pagedetails.ac...,404
4,http://www.gpo.gov/fdsys/search/pagedetails.ac...,404


In [ ]:
!mkdir -p EOT-2012/parquet/full/


In [ ]:
!aws s3 sync --no-sign-request s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/ EOT-2012/parquet/full/


download: s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00003-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet to EOT-2012/parquet/full/part-00003-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
download: s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00002-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet to EOT-2012/parquet/full/part-00002-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
download: s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet to EOT-2012/parquet/full/part-00001-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
download: s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00004-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet to EOT-2012/parquet/full/part-00004-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.parquet
download: s3://eotarchive/eot-index/table/eot-main/crawl=EOT-2012/part-00008-9a26f426-026b-4ca3-8e28-cbae857489be-c000.gz.pa

In [ ]:
duckdb.sql("""
    CREATE VIEW eot_2012 AS
    SELECT *
    FROM '/content/EOT-2012/parquet/full/**/*.gz.parquet'
""")

In [ ]:
duckdb.sql("""
    SELECT *
    FROM eot_2012
    WHERE fetch_status = 404
    LIMIT 5
""").df()

,url_surtkey,url,url_host_name,url_host_tld,url_host_2nd_last_part,url_host_3rd_last_part,url_host_4th_last_part,url_host_5th_last_part,url_host_registry_suffix,url_host_registered_domain,...,content_mime_detected,content_charset,content_languages,content_puid,warc_filename,warc_record_offset,warc_record_length,warc_segment,crawl,subset
0,"com,youtube)/embed/ttp:/www.youtube.com/v/e1ppg105qkg",http://www.youtube.com/embed/ttp://www.youtube.com/v/e1PPg105Qkg,www.youtube.com,com,youtube,www,None,None,com,youtube.com,...,text/html,ascii,eng,fmt/471,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2012-20120921164802214-01885-26126~wbgrp-crawl002.us.archive.org~8443.warc.gz,877256753,688,IA-000,EOT-2012,warc
1,"com,youtube)/embed/ttp:/www.youtube.com/v/e1ppg105qkg",http://www.youtube.com/embed/ttp://www.youtube.com/v/e1PPg105Qkg,www.youtube.com,com,youtube,www,None,None,com,youtube.com,...,text/html,ascii,eng,fmt/471,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2012-20121009231250245-01145-23849~wbgrp-crawl002.us.archive.org~8443.warc.gz,348478341,691,IA-000,EOT-2012,warc
2,"com,youtube)/embed/ttp:/www.youtube.com/v/e1ppg105qkg",http://www.youtube.com/embed/ttp://www.youtube.com/v/e1PPg105Qkg,www.youtube.com,com,youtube,www,None,None,com,youtube.com,...,text/html,ascii,eng,fmt/471,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2012-20121017093350314-00658-18439~wbgrp-crawl007.us.archive.org~8443.warc.gz,540914785,690,IA-000,EOT-2012,warc
3,"com,youtube)/embed/ttp:/www.youtube.com/v/e1ppg105qkg",http://www.youtube.com/embed/ttp://www.youtube.com/v/e1PPg105Qkg,www.youtube.com,com,youtube,www,None,None,com,youtube.com,...,text/html,ascii,eng,fmt/471,crawl-data/EOT-2012/segments/LOC-000/warc/LOC-EOT2012-001-20121108084543664-00227-15895~wbgrp-crawl012.us.archive.org~8443.warc.gz,902020843,689,LOC-000,EOT-2012,warc
4,"com,youtube)/embed/ttp:/www.youtube.com/v/e1ppg105qkg",http://www.youtube.com/embed/ttp://www.youtube.com/v/e1PPg105Qkg,www.youtube.com,com,youtube,www,None,None,com,youtube.com,...,text/html,ascii,eng,fmt/471,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2012-20130215200307899-00640-10476~wbgrp-crawl013.us.archive.org~8443.warc.gz,516945292,685,IA-000,EOT-2012,warc


In [ ]:
duckdb.sql("""
    SELECT
        url_host_2nd_last_part,
        COUNT(*) AS url_count
    FROM '/content/EOT-2012/parquet/full/**/*.gz.parquet'
    GROUP BY url_host_2nd_last_part
    ORDER BY url_count DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,url_host_2nd_last_part,url_count
0,house,36112972
1,senate,9967319
2,loc,7593918
3,facebook,7510625
4,dvidshub,5613295
...,...,...
170105,what,1
170106,univ-provence,1
170107,wonb,1
170108,wpat,1


In [ ]:
duckdb.sql("""
    SELECT
        url_host_private_suffix,
        COUNT(*) AS url_count
    FROM '/content/EOT-2012/parquet/full/**/*.gz.parquet'
    GROUP BY url_host_private_suffix
    ORDER BY url_count DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,url_host_private_suffix,url_count
0,gov,109889349
1,com,45830183
2,mil,16320040
3,net,9155981
4,org,5718069
...,...,...
1086,ec2-50-17-84-46.compute-1.amazonaws.com,1
1087,ec2-72-44-63-165.compute-1.amazonaws.com,1
1088,ec2-75-101-197-7.compute-1.amazonaws.com,1
1089,no.com,1


## Stream all .cdxj into parquet and then into duckdb

In [ ]:
!mkdir EOT-2012/converted/

In [ ]:
!aws s3 cp \
  --no-sign-request \
  --recursive \
  --exclude "*" \
  --include "*.cdxj.gz" \
  s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/ \
  EOT-2012/cdx/

Streaming output truncated to the last 5000 lines.
download: s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/EOT-2012-20121007081959839-00461-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz to EOT-2012/cdx/EOT-2012-20121007081959839-00461-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz
download: s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/EOT-2012-20121007082105231-00463-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz to EOT-2012/cdx/EOT-2012-20121007082105231-00463-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz
download: s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/EOT-2012-20121007083029722-00467-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz to EOT-2012/cdx/EOT-2012-20121007083029722-00467-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz
download: s3://eotarchive/crawl-data/EOT-2012/segments/IA-000/cdx/EOT-2012-20121007081257359-00459-23849~wbgrp-crawl002.us.archive.org~8443.cdxj.gz to EOT-2012/cdx/EOT-2012-20121007081257359-00459-23849~wbgrp-cr

In [ ]:
#

In [ ]:
import os, glob, gzip, json
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

paths = sorted(glob.glob("/content/EOT-2012/cdx/*.cdxj.gz"))
out_dir = "/content/EOT-2012/converted"
os.makedirs(out_dir, exist_ok=True)

# Choose the columns you want to keep (add/remove as needed)
COLUMNS = [
    "url", "mime", "status", "digest", "length", "offset", "filename",
    "mime-detected", "puid", "charset", "languages",
    "surtkey", "timestamp"
]

BATCH_SIZE = 50_000

for path in tqdm(paths, desc="Converting shards"):
    out = os.path.basename(path).replace(".cdxj.gz", ".parquet")
    out_path = os.path.join(out_dir, out)

    writer = None
    batch = []

    def normalize(rec):
        # ensure all columns exist
        return {k: rec.get(k) for k in COLUMNS}

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            try:
                key, ts, json_blob = line.strip().split(" ", 2)
                rec = json.loads(json_blob)
                rec["surtkey"] = key
                rec["timestamp"] = ts
                batch.append(normalize(rec))
            except:
                continue

            if len(batch) >= BATCH_SIZE:
                table = pa.Table.from_pylist(batch)
                if writer is None:
                    writer = pq.ParquetWriter(
                        out_path,
                        table.schema,
                        compression="zstd",
                        use_dictionary=True
                    )
                writer.write_table(table)
                batch.clear()

    if batch:
        table = pa.Table.from_pylist(batch)
        if writer is None:
            writer = pq.ParquetWriter(
                out_path,
                table.schema,
                compression="zstd",
                use_dictionary=True
            )
        writer.write_table(table)

    if writer is not None:
        writer.close()


Converting shards:   0%|          | 0/10000 [00:00<?, ?it/s]

In [ ]:
import duckdb, os
con = duckdb.connect()  # or duckdb.connect("/content/eot.duckdb")
con.execute(f"PRAGMA threads={os.cpu_count() or 4};")
con.execute("PRAGMA enable_progress_bar;")

In [ ]:
# only read url column to speed things up...
con.execute("""
CREATE OR REPLACE TABLE eot_hosts AS
SELECT
  lower(regexp_extract(url, '^[a-z]+://([^/?#:]+)', 1)) AS host
FROM parquet_scan('/content/EOT-2012/converted/*.parquet');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql("SELECT COUNT(*) FROM eot_hosts").show()
con.sql("SELECT * FROM eot_hosts LIMIT 5").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     48681429 │
└──────────────┘

┌──────────────────────┐
│         host         │
│       varchar        │
├──────────────────────┤
│ www.afcrossroads.com │
│ www.afcrossroads.com │
│ www.afcrossroads.com │
│ www.afcrossroads.com │
│ www.afcrossroads.com │
└──────────────────────┘



In [ ]:
#use ends_with which si faster than regex
con.sql("""
WITH labeled AS (
  SELECT
    host,
    CASE
      WHEN host='usda.gov' OR ends_with(host, '.usda.gov') THEN 'usda.gov'
      WHEN host='commerce.gov' OR ends_with(host, '.commerce.gov') THEN 'commerce.gov'
      WHEN host='defense.gov' OR ends_with(host, '.defense.gov') THEN 'defense.gov'
      WHEN host='ed.gov' OR ends_with(host, '.ed.gov') THEN 'ed.gov'
      WHEN host='energy.gov' OR ends_with(host, '.energy.gov') THEN 'energy.gov'
      WHEN host='hhs.gov' OR ends_with(host, '.hhs.gov') THEN 'hhs.gov'
      WHEN host='dhs.gov' OR ends_with(host, '.dhs.gov') THEN 'dhs.gov'
      WHEN host='hud.gov' OR ends_with(host, '.hud.gov') THEN 'hud.gov'
      WHEN host='doi.gov' OR ends_with(host, '.doi.gov') THEN 'doi.gov'
      WHEN host='justice.gov' OR ends_with(host, '.justice.gov') THEN 'justice.gov'
      WHEN host='dol.gov' OR ends_with(host, '.dol.gov') THEN 'dol.gov'
    END AS base_domain
  FROM eot_hosts
)
SELECT
  base_domain,
  host,
  COUNT(*) AS captures
FROM labeled
WHERE base_domain IS NOT NULL
GROUP BY base_domain, host
ORDER BY base_domain, captures DESC;
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,base_domain,host,captures
0,commerce.gov,2001-2009.commerce.gov,63769
1,commerce.gov,www.commerce.gov,47720
2,commerce.gov,selectusa.commerce.gov,3544
3,commerce.gov,recovery.commerce.gov,1036
4,commerce.gov,hr.commerce.gov,932
...,...,...,...
1127,usda.gov,agcensus.usda.gov,2
1128,usda.gov,pubsearch.arsnet.usda.gov,1
1129,usda.gov,google.search.csrees.usda.gov,1
1130,usda.gov,d2.nal.usda.gov,1


In [ ]:

con.sql("""
CREATE OR REPLACE VIEW eot_2012 AS
SELECT *
FROM read_parquet('/content/EOT-2012/converted/*.parquet');
""")

In [ ]:
con.sql("SELECT * FROM eot_2012 LIMIT 5").df()

,url,mime,status,digest,length,offset,filename,mime-detected,puid,charset,languages,surtkey,timestamp
0,http://www.afcrossroads.com/disclaimer.cfm?loc...,text/html,302,sha1:BSLFTGW4QCBGHGLKFKMYJLUBHTEI3Q5B,511,747376311,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2...,text/plain,None,ascii,None,"com,afcrossroads)/disclaimer.cfm?faqid=290&loc...",20120915075744
1,http://www.afcrossroads.com/disclaimer.cfm?loc...,text/html,302,sha1:BSLFTGW4QCBGHGLKFKMYJLUBHTEI3Q5B,484,570737348,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2...,text/plain,None,ascii,None,"com,afcrossroads)/disclaimer.cfm?location=http...",20120915075542
2,http://www.afcrossroads.com/disclaimer.cfm?loc...,text/html,302,sha1:BSLFTGW4QCBGHGLKFKMYJLUBHTEI3Q5B,447,455688148,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2...,text/plain,None,ascii,None,"com,afcrossroads)/disclaimer.cfm?location=http...",20120915075408
3,http://www.afcrossroads.com/disclaimer.cfm?loc...,text/html,302,sha1:BSLFTGW4QCBGHGLKFKMYJLUBHTEI3Q5B,500,722233435,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2...,text/plain,None,ascii,None,"com,afcrossroads)/disclaimer.cfm?location=http...",20120915075723
4,http://www.afcrossroads.com/disclaimer.cfm?loc...,text/html,302,sha1:BSLFTGW4QCBGHGLKFKMYJLUBHTEI3Q5B,478,681360811,crawl-data/EOT-2012/segments/IA-000/warc/EOT-2...,text/plain,None,ascii,None,"com,afcrossroads)/disclaimer.cfm?location=http...",20120915075655


In [ ]:
con.sql("SELECT COUNT(*) FROM eot_2012").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     48681429 │
└──────────────┘



In [ ]:
con.sql("""
SELECT COUNT(*)
FROM eot_2012
WHERE lower(url) LIKE '%justice.gov%';
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        64687 │
└──────────────┘



In [ ]:
con.sql("""
WITH hosts AS (
  SELECT
    lower(regexp_extract(url, '^[a-z]+://([^/?#:]+)', 1)) AS host
  FROM eot_2012
),
labeled AS (
  SELECT
    host,
    CASE
      WHEN host = 'usda.gov' OR host LIKE '%.usda.gov' THEN 'usda.gov'
      WHEN host = 'commerce.gov' OR host LIKE '%.commerce.gov' THEN 'commerce.gov'
      WHEN host = 'defense.gov' OR host LIKE '%.defense.gov' THEN 'defense.gov'
      WHEN host = 'ed.gov' OR host LIKE '%.ed.gov' THEN 'ed.gov'
      WHEN host = 'energy.gov' OR host LIKE '%.energy.gov' THEN 'energy.gov'
      WHEN host = 'hhs.gov' OR host LIKE '%.hhs.gov' THEN 'hhs.gov'
      WHEN host = 'dhs.gov' OR host LIKE '%.dhs.gov' THEN 'dhs.gov'
      WHEN host = 'hud.gov' OR host LIKE '%.hud.gov' THEN 'hud.gov'
      WHEN host = 'doi.gov' OR host LIKE '%.doi.gov' THEN 'doi.gov'
      WHEN host = 'justice.gov' OR host LIKE '%.justice.gov' THEN 'justice.gov'
      WHEN host = 'dol.gov' OR host LIKE '%.dol.gov' THEN 'dol.gov'
    END AS base_domain
  FROM hosts
)
SELECT
  base_domain,
  host,
  COUNT(*) AS captures
FROM labeled
WHERE base_domain IS NOT NULL
GROUP BY base_domain, host
ORDER BY base_domain, captures DESC;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,base_domain,host,captures
0,commerce.gov,2001-2009.commerce.gov,63769
1,commerce.gov,www.commerce.gov,47720
2,commerce.gov,selectusa.commerce.gov,3544
3,commerce.gov,recovery.commerce.gov,1036
4,commerce.gov,hr.commerce.gov,932
...,...,...,...
1127,usda.gov,techprs.sc.egov.usda.gov,2
1128,usda.gov,d2.nal.usda.gov,1
1129,usda.gov,offices.usda.gov,1
1130,usda.gov,pubsearch.arsnet.usda.gov,1


In [ ]:
con.sql("""CREATE TABLE eot_paths AS
SELECT
  lower(regexp_extract(url,'^[a-z]+://([^/]+)',1)) AS host,
  regexp_extract(url,'^[a-z]+://[^/]+(/[^?#]*)',1) AS path
FROM eot_2012;""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql("""
SELECT
  host,
  COALESCE(NULLIF(list_extract(string_split(trim(path, '/'), '/'), 1), ''), '(root)') AS path1,
  COUNT(*) AS captures
FROM eot_paths
WHERE host = 'commerce.gov' OR ends_with(host, '.commerce.gov')
GROUP BY host, path1
ORDER BY captures DESC;
""").df()

,host,path1,captures
0,2001-2009.commerce.gov,opa,50926
1,www.commerce.gov,blog,15799
2,www.commerce.gov,news,8311
3,www.commerce.gov,sites,7642
4,2001-2009.commerce.gov,NewsRoom,7258
...,...,...,...
356,www.commerce.gov,evans_bio.html,1
357,selectusa.commerce.gov,%E2%80%9Dhttp%3A,1
358,acetool.my.commerce.gov,misc,1
359,acetool.my.commerce.gov,news,1


In [ ]:
con.sql("""
WITH labeled AS (
  SELECT
    host,
    path,
    CASE
      WHEN host='usda.gov' OR ends_with(host,'.usda.gov') THEN 'usda.gov'
      WHEN host='commerce.gov' OR ends_with(host,'.commerce.gov') THEN 'commerce.gov'
      WHEN host='defense.gov' OR ends_with(host,'.defense.gov') THEN 'defense.gov'
      WHEN host='ed.gov' OR ends_with(host,'.ed.gov') THEN 'ed.gov'
      WHEN host='energy.gov' OR ends_with(host,'.energy.gov') THEN 'energy.gov'
      WHEN host='hhs.gov' OR ends_with(host,'.hhs.gov') THEN 'hhs.gov'
      WHEN host='dhs.gov' OR ends_with(host,'.dhs.gov') THEN 'dhs.gov'
      WHEN host='hud.gov' OR ends_with(host,'.hud.gov') THEN 'hud.gov'
      WHEN host='doi.gov' OR ends_with(host,'.doi.gov') THEN 'doi.gov'
      WHEN host='justice.gov' OR ends_with(host,'.justice.gov') THEN 'justice.gov'
      WHEN host='dol.gov' OR ends_with(host,'.dol.gov') THEN 'dol.gov'
    END AS base_domain
  FROM eot_paths
)
SELECT
  base_domain,
  host,
  COALESCE(NULLIF(list_extract(string_split(trim(path, '/'), '/'), 1), ''), '(root)') AS path1,
  COUNT(*) AS captures
FROM labeled
WHERE base_domain IS NOT NULL
GROUP BY base_domain, host, path1
ORDER BY base_domain, host, captures DESC;
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,base_domain,host,path1,captures
0,commerce.gov,2001-2009.commerce.gov,opa,50926
1,commerce.gov,2001-2009.commerce.gov,NewsRoom,7258
2,commerce.gov,2001-2009.commerce.gov,s,4144
3,commerce.gov,2001-2009.commerce.gov,Services,539
4,commerce.gov,2001-2009.commerce.gov,graphics,227
...,...,...,...,...
26082,usda.gov,www3.wcc.nrcs.usda.gov,reportGenerator,31
26083,usda.gov,www3.wcc.nrcs.usda.gov,robots.txt,4
26084,usda.gov,www3.wcc.nrcs.usda.gov,siteimages,2
26085,usda.gov,www3.wcc.nrcs.usda.gov,contact,1


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE eot_host_path2 AS
SELECT
  CASE
    WHEN host='usda.gov' OR ends_with(host,'.usda.gov') THEN 'usda.gov'
    WHEN host='commerce.gov' OR ends_with(host,'.commerce.gov') THEN 'commerce.gov'
    WHEN host='defense.gov' OR ends_with(host,'.defense.gov') THEN 'defense.gov'
    WHEN host='ed.gov' OR ends_with(host,'.ed.gov') THEN 'ed.gov'
    WHEN host='energy.gov' OR ends_with(host,'.energy.gov') THEN 'energy.gov'
    WHEN host='hhs.gov' OR ends_with(host,'.hhs.gov') THEN 'hhs.gov'
    WHEN host='dhs.gov' OR ends_with(host,'.dhs.gov') THEN 'dhs.gov'
    WHEN host='hud.gov' OR ends_with(host,'.hud.gov') THEN 'hud.gov'
    WHEN host='doi.gov' OR ends_with(host,'.doi.gov') THEN 'doi.gov'
    WHEN host='justice.gov' OR ends_with(host,'.justice.gov') THEN 'justice.gov'
    WHEN host='dol.gov' OR ends_with(host,'.dol.gov') THEN 'dol.gov'
  END AS base_domain,

  host,

  CASE
    WHEN path IS NULL OR path='' OR path='/' THEN '(root)'
    ELSE array_to_string(
      list_slice(string_split(trim(path,'/'),'/'), 1, 2),
      '/'
    )
  END AS path2

FROM eot_paths
WHERE
  host='usda.gov' OR ends_with(host,'.usda.gov')
  OR host='commerce.gov' OR ends_with(host,'.commerce.gov')
  OR host='defense.gov' OR ends_with(host,'.defense.gov')
  OR host='ed.gov' OR ends_with(host,'.ed.gov')
  OR host='energy.gov' OR ends_with(host,'.energy.gov')
  OR host='hhs.gov' OR ends_with(host,'.hhs.gov')
  OR host='dhs.gov' OR ends_with(host,'.dhs.gov')
  OR host='hud.gov' OR ends_with(host,'.hud.gov')
  OR host='doi.gov' OR ends_with(host,'.doi.gov')
  OR host='justice.gov' OR ends_with(host,'.justice.gov')
  OR host='dol.gov' OR ends_with(host,'.dol.gov');
""")
con.execute("CREATE INDEX IF NOT EXISTS idx_host_path2 ON eot_host_path2(host);")
con.execute("CREATE INDEX IF NOT EXISTS idx_base_domain ON eot_host_path2(base_domain);")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.sql("""
SELECT base_domain, host, path2, COUNT(*) AS captures
FROM eot_host_path2
GROUP BY base_domain, host, path2
ORDER BY base_domain, host, captures DESC;
""").df()

,base_domain,host,path2,captures
0,commerce.gov,2001-2009.commerce.gov,opa/photo,50870
1,commerce.gov,2001-2009.commerce.gov,s/groups,4102
2,commerce.gov,2001-2009.commerce.gov,NewsRoom/PressReleases_FactSheets,2810
3,commerce.gov,2001-2009.commerce.gov,NewsRoom/TopNews,2197
4,commerce.gov,2001-2009.commerce.gov,NewsRoom/SecretarySpeeches,1268
...,...,...,...,...
119626,usda.gov,www3.wcc.nrcs.usda.gov,nwcc/dscan.js,1
119627,usda.gov,www3.wcc.nrcs.usda.gov,contact/db-message.html,1
119628,usda.gov,www3.wcc.nrcs.usda.gov,siteimages/413.jpg,1
119629,usda.gov,www3.wcc.nrcs.usda.gov,nwcc/dscan.css,1
